In [2]:
%matplotlib inline

import pyvista as pv
import glob
import os
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.animation import FuncAnimation
import numpy as np
import math
from matplotlib.animation import FuncAnimation, PillowWriter
from scipy.interpolate import interp1d

mpl.rcParams['figure.figsize'] = (15,10)
mpl.rcParams['axes.labelsize'] = 30
mpl.rcParams['axes.titlesize'] = 30
mpl.rcParams['xtick.labelsize'] = 20
mpl.rcParams['ytick.labelsize'] = 20
mpl.rcParams['legend.fontsize'] = 20
mpl.rcParams['lines.linewidth'] = 5

data_folder = 'data/free_zoom_lowerecho/0.33603529607296023/evolve'  # folder containing all the VTUs for a single run

crit_time = 6.547
frame_interval = 0.03

## Import the VTU files

In [3]:
# make a dictionary with all the field values
field_data = {}
# loop through VTUs in the folder
for file in glob.glob(data_folder+'/*'):
    # make a new dictionary for this time step
    data_time_step = {}
    # get the frame number from the filename
    time_index = int(file.split('_')[-1].split('.')[0])
    # read in the VTU
    vtu = pv.read(file)
    # extract radius, Phi, and lapse values at this time step
    data_time_step['r'] = vtu.points[:,0]
    data_time_step['Phi'] = vtu.point_data['Fields::Phi']
    data_time_step['lapse'] = vtu.point_data['Fields::Lapse']
    # calculate phi (the field) from Phi (the gradient of the field)
    data_time_step['phi'] = np.flip(np.array([ np.trapezoid(np.flip(data_time_step['Phi'])[:end+1], np.flip(data_time_step['r'])[:end+1]) for end in range(len(data_time_step['Phi'])) ]))
    # add this time step into the full data
    field_data[time_index] = data_time_step
    
# calculate some other important quantities and add it into the data
for tt in np.sort(list(field_data.keys())):
    # calculate proper time from coordinate time, lapse, and frame_interval (defined above)
    proper_time = field_data[tt]['lapse'][0]*frame_interval+field_data[tt-1]['proper_time'] if tt!=0 else field_data[tt]['lapse'][0]*frame_interval
    field_data[tt]['proper_time'] = proper_time
    # calculate log time (tau)
    field_data[tt]['tau'] = -np.log(crit_time-proper_time)

In [4]:
# how many frames in our data?
len(field_data.keys())

975

## Animate

In [5]:
%matplotlib qt

# set up the figure and empty artists
fig, ax = plt.subplots(figsize=(13,5))
profile, = ax.plot([], [], c='black', label=f"$-\\tau=$ {field_data[0]['tau']}")
plt.tight_layout()

# initialize the plot
def init():
    ax.set_ylim(-0.15,0.15)
    ax.set_xlim(-6.,2.)
    ax.set_ylabel(r"$r \phi$'")
    ax.set_xlabel(r'log radius ($\rho$)')
    plt.tight_layout()
    ax.legend(loc='upper left')
    ax.grid()

# animation step
def update(time):
    # show the field
    profile.set_data(np.log(field_data[time]['r']), field_data[time]['r']*field_data[time]['Phi'])
    # update the legend text to show current time
    profile.set_label(f"$-\\tau=$ {field_data[time]['tau']:.3f}")
    ax.legend(loc='upper left')
    plt.tight_layout()
        
# set up log frames (so the animation slows down near the end)
frames = np.sort(list(field_data.keys()))
u = np.linspace(1, 10, len(frames))
mapped = np.log(u) / np.log(u[-1])
log_frames = (mapped * (frames[-1] - frames[0]) + frames[0]).astype(int)
log_frames = log_frames[log_frames <= 970]
log_frames = np.concatenate([log_frames, np.full(20, log_frames[-1])])

# plot
anim = FuncAnimation(fig, update, init_func=init, frames=log_frames, interval=15)
plt.show()
# writer = PillowWriter(fps=40)
# anim.save("/home/ben/Desktop/swigart_collapse_40fps.gif", writer=writer)

/tmp/ipykernel_23131/3445356070.py:19: RuntimeWarning: divide by zero encountered in log
  profile.set_data(np.log(field_data[time]['r']), field_data[time]['r']*field_data[time]['Phi'])
